In [1]:
!git clone https://github.com/vfxbymani/ai-localization-pipeline.git
%cd ai-localization-pipeline
!ls

Cloning into 'ai-localization-pipeline'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 46 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 1.31 MiB | 5.14 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/kaggle/working/ai-localization-pipeline
day01-pytorch-basics.ipynb	   day05-whisper-wer.ipynb     README.md
day02-transfer-learning.ipynb	   day06-video-to-srt.ipynb    test_video.srt
day03-huggingface-basics.ipynb	   day07-tamil-pipeline.ipynb  video_to_srt.py
day04-whisper-transcription.ipynb  LOG.md


In [2]:
import re

def parse_srt(path):
    text = open(path, encoding="utf-8").read().strip()
    cues = []
    for block in re.split(r"\n\s*\n", text):
        lines = block.strip().splitlines()
        if len(lines) < 3:
            continue
        start, end = [t.strip() for t in lines[1].split("-->")]
        cues.append({
            "index": int(lines[0]),
            "start": start,
            "end": end,
            "text": " ".join(lines[2:]),
        })
    return cues

cues = parse_srt("test_video.srt")
print("cues:", len(cues))
for c in cues[:3]:
    print(c)

cues: 36
{'index': 1, 'start': '00:00:00,000', 'end': '00:00:03,988', 'text': 'Mr. Quilter is the apostle of the middle classes,'}
{'index': 2, 'start': '00:00:03,988', 'end': '00:00:07,080', 'text': 'and we are glad to welcome his gospel.'}
{'index': 3, 'start': '00:00:07,080', 'end': '00:00:12,280', 'text': "Nor is Mr. Quilter's manner less interesting than his matter."}


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(name).to("cuda")
tamil_id = tokenizer.convert_tokens_to_ids("tam_Taml")

def translate(texts, batch_size=8):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            ids = model.generate(**inputs, forced_bos_token_id=tamil_id, max_new_tokens=128)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

english = [c["text"] for c in cues]
tamil = translate(english)

translated = [dict(c, text=t) for c, t in zip(cues, tamil)]

print("translated cues:", len(translated))
print("timings unchanged:", all(a["start"] == b["start"] and a["end"] == b["end"] for a, b in zip(cues, translated)))

lens = [len(t) for t in tamil]
ratio = sum(lens) / sum(len(e) for e in english)
print("average Tamil/English length ratio:", round(ratio, 2))
print("longest Tamil cue (chars):", max(lens))
print("cues over 84 characters (2 lines x 42):", sum(l > 84 for l in lens))
print()
for e, t in list(zip(english, tamil))[:6]:
    print(e)
    print(t)
    print()

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

translated cues: 36
timings unchanged: True
average Tamil/English length ratio: 1.13
longest Tamil cue (chars): 108
cues over 84 characters (2 lines x 42): 3

Mr. Quilter is the apostle of the middle classes,
திரு Quilter நடுத்தர வர்க்கத்தின் அப்போஸ்தலன்,

and we are glad to welcome his gospel.
அவருடைய நற்செய்தியை ஏற்றுக் கொண்டதில் நாங்கள் மகிழ்ச்சியடைகிறோம்.

Nor is Mr. Quilter's manner less interesting than his matter.
திரு Quilter இன் முறை அவரது விவகாரம் விட குறைவான சுவாரஸ்யமான இல்லை.

He tells us that at this festive season of the year,
அவர் நமக்கு சொல்கிறார் இந்த பண்டிகை காலத்தில்,

with Christmas and roast beef looming
கிறிஸ்துமஸ் மற்றும் வறுத்த மாட்டிறைச்சி வரப்போகிறது

before us, similes drawn from eating and
நமக்கு முன்னால், சாப்பிடுவதன் மூலமும்,



In [4]:
from video_to_srt import split_text, fit_lines

ABBREV = {"mr.", "mrs.", "ms.", "dr.", "st."}

def ends_sentence(text):
    last = text.split()[-1]
    return last[-1] in ".?!" and last.lower() not in ABBREV

# 1. group consecutive cues into whole sentences
groups, cur = [], []
for c in cues:
    cur.append(c)
    if ends_sentence(c["text"]):
        groups.append(cur)
        cur = []
if cur:
    groups.append(cur)

# 2. translate each whole sentence
def translate(texts, batch_size=8, max_new_tokens=256):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            ids = model.generate(**inputs, forced_bos_token_id=tamil_id, max_new_tokens=max_new_tokens)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

sentences = [" ".join(c["text"] for c in g) for g in groups]
tamil_sent = translate(sentences)

# 3. spread each Tamil sentence back over the original cue timings
new_cues = []
for g, ta in zip(groups, tamil_sent):
    n = len(g)
    parts = split_text(ta, n) if n > 1 else [ta]
    if len(parts) != n:
        new_cues.append({"start": g[0]["start"], "end": g[-1]["end"], "text": ta})
        continue
    for c, p in zip(g, parts):
        new_cues.append({"start": c["start"], "end": c["end"], "text": p})

def write_srt(cs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, c in enumerate(cs, start=1):
            lines = fit_lines(c["text"])
            f.write(f"{i}\n{c['start']} --> {c['end']}\n" + "\n".join(lines) + "\n\n")

write_srt(new_cues, "test_video_tamil.srt")

lens = [len(c["text"]) for c in new_cues]
print("sentence groups:", len(groups), "| cues:", len(new_cues))
print("cues over 84 characters:", sum(l > 84 for l in lens))
print("longest cue (chars):", max(lens))
print()
for c in new_cues[:6]:
    print(c["start"], "->", c["end"])
    print(c["text"])
    print()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sentence groups: 14 | cues: 36
cues over 84 characters: 3
longest cue (chars): 108

00:00:00,000 -> 00:00:03,988
திரு. குயில்டர் நடுத்தர வர்க்கத்தின் அப்போஸ்தலர்,

00:00:03,988 -> 00:00:07,080
அவருடைய நற்செய்தியை வரவேற்பதில் நாங்கள் மகிழ்ச்சியடைகிறோம்.

00:00:07,080 -> 00:00:12,280
திரு Quilter இன் முறை அவரது விவகாரம் விட குறைவான சுவாரஸ்யமான இல்லை.

00:00:12,280 -> 00:00:15,529
கிறிஸ்துமஸ் மற்றும் வறுத்த இறைச்சிகள் நமக்கு

00:00:15,529 -> 00:00:17,840
முன்னால் வரும் இந்த பண்டிகை காலத்தில்,

00:00:17,840 -> 00:00:21,426
உணவு மற்றும் அதன் விளைவுகள் போன்றவை மனதில் மிக



In [5]:
import re
from video_to_srt import split_text, fmt_time

cues_step3 = new_cues

def to_sec(t):
    h, m, rest = t.split(":")
    s, ms = rest.split(",")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def clause_cues(g, ta, max_chars=84):
    pieces = [p for p in re.split(r"(?<=[,;.?!])\s+", ta.strip()) if p]
    packed, cur = [], ""
    for p in pieces:
        if cur and len(cur) + 1 + len(p) > max_chars:
            packed.append(cur)
            cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur:
        packed.append(cur)
    final = []
    for p in packed:
        final += split_text(p, 2) if len(p) > max_chars else [p]
    start, end = to_sec(g[0]["start"]), to_sec(g[-1]["end"])
    total = sum(len(p) for p in final)
    out, t = [], start
    for p in final:
        e = t + (end - start) * len(p) / total
        out.append({"start": fmt_time(t), "end": fmt_time(e), "text": p})
        t = e
    return out

cues_step4 = []
for g, ta in zip(groups, tamil_sent):
    cues_step4 += clause_cues(g, ta)

def stats(cs):
    durs = [to_sec(c["end"]) - to_sec(c["start"]) for c in cs]
    punct = sum(c["text"].strip()[-1] in ",;.?!" for c in cs)
    return {
        "cues": len(cs),
        "over 84 chars": sum(len(c["text"]) > 84 for c in cs),
        "ending at punctuation": f"{punct}/{len(cs)}",
        "shortest (s)": round(min(durs), 2),
        "longest (s)": round(max(durs), 2),
    }

print("step 3:", stats(cues_step3))
print("step 4:", stats(cues_step4))

write_srt(cues_step4, "test_video_tamil_clauses.srt")
print()
for c in cues_step4[:8]:
    print(c["start"], "->", c["end"])
    print(c["text"])
    print()

step 3: {'cues': 36, 'over 84 chars': 3, 'ending at punctuation': '23/36', 'shortest (s)': 0.84, 'longest (s)': 6.68}
step 4: {'cues': 32, 'over 84 chars': 2, 'ending at punctuation': '26/32', 'shortest (s)': 0.69, 'longest (s)': 7.25}

00:00:00,000 -> 00:00:03,212
திரு. குயில்டர் நடுத்தர வர்க்கத்தின் அப்போஸ்தலர்,

00:00:03,212 -> 00:00:07,080
அவருடைய நற்செய்தியை வரவேற்பதில் நாங்கள் மகிழ்ச்சியடைகிறோம்.

00:00:07,080 -> 00:00:12,280
திரு Quilter இன் முறை அவரது விவகாரம் விட குறைவான சுவாரஸ்யமான இல்லை.

00:00:12,280 -> 00:00:18,481
கிறிஸ்துமஸ் மற்றும் வறுத்த இறைச்சிகள் நமக்கு முன்னால் வரும் இந்த பண்டிகை காலத்தில்,

00:00:18,481 -> 00:00:21,918
உணவு மற்றும் அதன் விளைவுகள் போன்றவை மனதில் மிக

00:00:21,918 -> 00:00:25,280
எளிதாக தோன்றும் என்று அவர் நமக்கு சொல்கிறார்.

00:00:25,280 -> 00:00:27,794
சர் ஃபிரடரிக் லெய்டனின் படைப்புகள் உண்மையில் கிரேக்கமானதா

00:00:27,794 -> 00:00:30,000
என்பது குறித்து அவருக்கு கடுமையான சந்தேகம் உள்ளது.

